In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Myriad Pro'
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 600

import numpy as np
import scipy.stats as stats

from scipy.stats import kstest, cramervonmises
# import tensorflow as tf
# import tensorflow_probability as tfp
import pykrige.kriging_tools as kt
from pykrige.ok import OrdinaryKriging
import time

from ipcc_colormap import *
from utils import *

from matplotlib import gridspec
from matplotlib.colorbar import ColorbarBase

import pickle

coastline = gpd.read_file('coastlines-split-SGregion/lines.shp')
mask = np.loadtxt('mask.txt')

ipcc_blue = (112.0/255, 160.0/255, 205.0/255, 1.0)
ipcc_orange = (196.0/255, 121.0/255, 0.0/255, 1.0)

tmp_cmap = ipcc_cmap()
tmp_cmap.read_rgb_data_from_excel()
;

Matplotlib is building the font cache; this may take a moment.


''

In [2]:
# DATA PREPARATION
# Load historical data
rain_obs = np.loadtxt('data/sta_monthly.csv')
rain_sim_flatten = np.loadtxt('data/wrf_monthly.csv')
rain_sim = rain_sim_flatten.reshape(rain_sim_flatten.shape[0], 120, 160)

# Station location mapping
sim_sel = np.loadtxt('data/wrf_loc.csv')
sim_idx = sim_sel[:, :2].astype(int)
sta_loc = np.genfromtxt('data/sta_lookup_new.csv', delimiter=',')[:, 2:]

# Grid coordinates
longlat = np.loadtxt('data/lonlat.txt')
lons = longlat[0, :].reshape(120, 160)
lats = longlat[1, :].reshape(120, 160)

# Historical WRF at station locations
wrf_sta_hist = np.array([rain_sim[:, i, j] for (i, j) in sim_idx]).T

# Load and process CMIP6-forced future climate data
cmip_rain_raw = np.loadtxt('data/wrf_monthly_cmip.txt')[-40:, :]
cmip_rain_temp = cmip_rain_raw.reshape(40, 12, 120, 160)

# Shift calendar: July starts CMIP simulations -> align with Jan-Dec
cmip_rain_aligned = np.zeros(cmip_rain_temp.shape)
cmip_rain_aligned[:, 0:6, :, :] = cmip_rain_temp[:, 6:12, :, :]  # Jan-Jun from Jul-Dec
cmip_rain_aligned[:, 6:12, :, :] = cmip_rain_temp[:, 0:6, :, :]  # Jul-Dec from Jan-Jun

# Future WRF data
cmip_rain_flatten = np.reshape(cmip_rain_aligned, (40, 12, 120*160))
wrf_sta_future = np.zeros((40, 12, sim_idx.shape[0]))
for idx, (i, j) in enumerate(sim_idx):
    wrf_sta_future[:, :, idx] = cmip_rain_aligned[:, :, i, j]

# Configuration constants
N_STATIONS = sim_idx.shape[0]  # 14 stations
N_MONTHS = 12
N_YEARS = 40
GRID_WIDTH = 120
GRID_HEIGHT = 160

FIGURE_DIR = 'figures'
HISTORICAL_PICKLE = 'intermediate/his_interpolation.pkl'

# Plotting setup
month_labels = ['(a) Jan', '(b) Feb', '(c) Mar', '(d) Apr', '(e) May', '(f) Jun', 
                '(g) Jul', '(h) Aug', '(i) Sep', '(j) Oct', '(k) Nov', '(l) Dec']

# Load historical interpolated rainfall for comparison
print("Loading historical interpolated rainfall...")
with open(HISTORICAL_PICKLE, 'rb') as pkl_file:
    rain_historical_interpolated = pickle.load(pkl_file)
print(f"Historical data shape: {rain_historical_interpolated.shape}")

Loading historical interpolated rainfall...
Historical data shape: (480, 120, 160)


In [3]:
cmip_rain_aligned.shape

(40, 12, 120, 160)

In [4]:
def counterfactual_climate_interpolation(month_idx, rain_obs, wrf_sta_hist, wrf_sta_future, 
                                       cmip_rain_flatten, rain_sim_flatten):
    """
    Perform counterfactual climate interpolation for a specific month.
    
    Uses historical means but future covariances to isolate the effect of 
    changed spatial correlation patterns under climate change.
    
    Args:
        month_idx: Month index (0-11)
        rain_obs: Historical station observations
        wrf_sta_hist: Historical WRF at station locations
        wrf_sta_future: Future WRF at station locations  
        cmip_rain_flatten: Future WRF at all grid points
        rain_sim_flatten: Historical WRF at all grid points
        
    Returns:
        np.array: Interpolated rainfall (N_YEARS, GRID_WIDTH, GRID_HEIGHT)
    """
    print(f'Processing {month_labels[month_idx]} - ', end='')
    start_time = time.time()
    
    # Extract monthly data
    month_obs = rain_obs[month_idx::N_MONTHS, :]
    month_wrf_hist = wrf_sta_hist[month_idx::N_MONTHS, :]
    month_wrf_future = wrf_sta_future[:, month_idx, :]
    month_cmip_grid = cmip_rain_flatten[:, month_idx, :]
    month_hist_grid = rain_sim_flatten[month_idx::N_MONTHS, :]
    
    # Step 1: Calibrate noise parameters using historical data
    gp_hist = gp_interpolator(P=N_STATIONS)
    gp_hist.read_rainfall(month_obs, month_wrf_hist)
    gp_hist.sn_converge()
    calibrated_noise = gp_hist.sn.copy()
    
    # Step 2: Compute historical means for counterfactual
    hist_mu_x = np.mean(month_wrf_hist, axis=0)  # Historical station means
    hist_mu_y = np.mean(month_hist_grid, axis=0)[:, None]  # Historical grid means
    fut_mu_x = np.mean(month_wrf_future, axis=0)
    fut_mu_y = np.mean(month_cmip_grid, axis=0)[:, None]
    
    print(f'Noise calibrated - ', end='')
    
    # Step 3: Create counterfactual GP with future covariances but historical means
    gp_counterfactual = gp_interpolator(P=N_STATIONS)
    gp_counterfactual.sn = calibrated_noise  # Use pre-calibrated noise
    
    # Read future data but override with historical means
    # gp_counterfactual.read_rainfall(month_obs, month_wrf_future, mu_x=hist_mu_x)
    gp_counterfactual.read_rainfall(month_obs, month_wrf_future, mu_x=fut_mu_x)
    
    # Step 4: Predict using future covariances but historical target means
    # predictions, _ = gp_counterfactual.predict(month_cmip_grid, mu_y=hist_mu_y)
    predictions, _ = gp_counterfactual.predict(month_cmip_grid, mu_y=fut_mu_y)
    
    # Reshape and ensure non-negative
    predictions_reshaped = np.reshape(predictions.T, (N_YEARS, GRID_WIDTH, GRID_HEIGHT))
    predictions_reshaped[predictions_reshaped < 0] = 0
    
    print(f'Time: {time.time() - start_time:.1f}s')
    
    return predictions_reshaped

In [5]:
def create_climate_comparison_plot(obs_means, historical_spatial, future_spatial, 
                                  mask, month_labels, output_path):
    """
    Create 12-panel comparison plot of historical vs future climate scenarios.
    
    Args:
        obs_means: List of observed station means for each month
        historical_spatial: List of historical spatial averages for each month  
        future_spatial: List of future spatial averages for each month
        mask: Spatial mask for averaging
        month_labels: Month labels for subplots
        output_path: Path to save the figure
    """
    fig, ax = plt.subplots(nrows=4, ncols=3, figsize=(9.5, 12))
    
    for month_idx in range(N_MONTHS):
        row_idx, col_idx = divmod(month_idx, 3)
        
        # Create scatter plots
        ax[row_idx][col_idx].scatter(obs_means[month_idx], historical_spatial[month_idx], 
                                   s=40, color=ipcc_blue, label='Historical (ERA5)', 
                                   alpha=0.67, edgecolors='None')
        ax[row_idx][col_idx].scatter(obs_means[month_idx], future_spatial[month_idx], 
                                   s=40, color=ipcc_orange, label='Future (CMIP6)', 
                                   alpha=0.67, edgecolors='None')
        
        # Add 1:1 line
        ax[row_idx][col_idx].axline([0, 0], [1, 1], color='black', linestyle='--')
        
        # Set axis limits and formatting
        x_min, x_max = np.min(obs_means[month_idx]), np.max(obs_means[month_idx])
        axis_min = max(0, x_min - 20)
        axis_max = x_max + 20
        ax[row_idx][col_idx].set_xlim([axis_min, axis_max])
        ax[row_idx][col_idx].set_ylim([axis_min, axis_max])
        
        # Format ticks
        ax[row_idx][col_idx].xaxis.set_major_locator(mticker.MultipleLocator(100))
        ax[row_idx][col_idx].yaxis.set_major_locator(mticker.MultipleLocator(100))
        ax[row_idx][col_idx].set_aspect('equal', adjustable='box')
        
        # Add month label
        ax[row_idx][col_idx].text(0.05, 0.95, month_labels[month_idx], 
                                transform=ax[row_idx][col_idx].transAxes, 
                                fontsize=10, verticalalignment='top', 
                                bbox=dict(facecolor='white', alpha=0.6))
        
        # Add legend to last subplot
        if month_idx == 11:
            ax[row_idx][col_idx].legend(loc='lower right')
    
    # Add axis labels
    fig.text(0.54, 0.04, 'Station-based Average Rainfall [mm]', ha='center', fontsize=14)
    fig.text(0.04, 0.53, 'Spatially Interpolated Average Rainfall [mm]', 
             va='center', rotation='vertical', fontsize=14)
    
    # Adjust layout and save
    fig.tight_layout(rect=[0.05, 0.05, 1, 1])
    fig.savefig(output_path, dpi=600, bbox_inches='tight')
    print(f'Saved comparison plot: {output_path}')
    
    return fig

In [6]:
def create_monthly_climatology_maps(interpolation_results, filename_suffix, 
                                   vmin=0, vmax=400, colorbar_label='Rainfall Climatology [mm]'):
    """
    Create 12-panel monthly climatology maps from interpolation results.
    
    Args:
        interpolation_results: List of 12 arrays (N_YEARS, GRID_WIDTH, GRID_HEIGHT)
        plot_title: Title for the figure
        filename_suffix: Suffix for output filename
        vmin, vmax: Color scale limits
        colorbar_label: Label for colorbar
    """
    fig = plt.figure(figsize=(11, 10))
    gs = gridspec.GridSpec(4, 3, height_ratios=[1,1,1,1], bottom=0.1, top=0.95, 
                          left=0.05, right=0.95, wspace=0.05, hspace=0.05)
    
    axes = [plt.subplot(gs[i, j], projection=crs.PlateCarree()) 
            for i in range(4) for j in range(3)]
    
    # Initialize plotter and colormap if not already done
    sg_plotter = sg_map_plotter(lons, lats, coastline)
    cmap = tmp_cmap.get_ipcc_cmap('seq', 'prec', 20)
    
    for month_idx, ax in enumerate(axes):
        # Calculate 40-year monthly mean
        monthly_mean = np.mean(interpolation_results[month_idx], axis=0)
        
        # Plot the climatology
        sg_plotter.plot_(ax, monthly_mean, cmap=cmap, vmin=vmin, vmax=vmax)
        sg_plotter.plot_scatter(ax, sta_loc, size=15)
        
        # Add month label
        ax.text(0.03, 0.95, month_labels[month_idx], transform=ax.transAxes, 
                fontsize=10, verticalalignment='top', 
                bbox=dict(facecolor='white', alpha=0.6))
    
    # Add colorbar
    cbar_ax = fig.add_axes([0.06, 0.07, 0.88, 0.015])
    bounds = np.linspace(vmin, vmax, 21)
    norm = mpl.colors.BoundaryNorm(bounds, cmap.N)
    cb = ColorbarBase(cbar_ax, cmap=cmap, norm=norm, orientation='horizontal')
    cb.ax.tick_params(labelsize=12)
    cb.set_label(colorbar_label, fontsize=12, fontweight='bold')
    
    # Save figure
    output_path = f'{FIGURE_DIR}/climatology_{filename_suffix}.pdf'
    fig.savefig(output_path, dpi=600, bbox_inches='tight')
    print(f'Saved climatology map: {output_path}')
    
    return fig

In [7]:
# Perform counterfactual climate analysis
print("Running counterfactual climate interpolation analysis...")
print("(Using future means and future covariances)\n")

# Storage for results
future_interpolated_results = []
obs_station_means = []
historical_spatial_means = []
future_spatial_means = []

for month_idx in range(N_MONTHS):
    # Counterfactual interpolation with corrected approach
    future_interpolated = counterfactual_climate_interpolation(
        month_idx, rain_obs, wrf_sta_hist, wrf_sta_future, 
        cmip_rain_flatten, rain_sim_flatten
    )
    future_interpolated_results.append(future_interpolated)
    
    # Calculate spatial averages for comparison
    month_obs_mean = np.mean(rain_obs[month_idx::N_MONTHS, :], axis=1)
    month_hist_spatial = np.nanmean(
        np.multiply(rain_historical_interpolated[month_idx::N_MONTHS, :, :], mask), 
        axis=(1, 2)
    )
    month_future_spatial = np.nanmean(
        np.multiply(future_interpolated, mask), 
        axis=(1, 2)
    )
    
    obs_station_means.append(month_obs_mean)
    historical_spatial_means.append(month_hist_spatial)
    future_spatial_means.append(month_future_spatial)

print(f"\nCounterfactual analysis complete!")

Running counterfactual climate interpolation analysis...
(Using future means and future covariances)

Processing (a) Jan - 1
[np.float64(68.46349707844365), np.float64(73.46137489975303), np.float64(77.26251277446661), np.float64(83.47726228031928), np.float64(76.084426989074), np.float64(101.90796438519892), np.float64(113.34412823210157), np.float64(77.69160544766474), np.float64(64.59860513227127), np.float64(51.185156071347926), np.float64(75.89352952321208), np.float64(70.31336710998535), np.float64(56.31201295582587), np.float64(64.6540826681844)]
2
[np.float64(44.87912534605323), np.float64(46.89171914710519), np.float64(43.52663083374642), np.float64(44.319963469110036), np.float64(84.13188722340738), np.float64(90.9049922809899), np.float64(80.12736076953206), np.float64(44.56162131010703), np.float64(51.267206391483676), np.float64(26.4925296373568), np.float64(44.74583184717229), np.float64(55.89306411837372), np.float64(43.20147756271526), np.float64(56.68873374377913)]
3
[

In [8]:
# Create comparison plot
print("\nCreating historical vs future climate comparison plot...")
output_path = f'{FIGURE_DIR}/counterfactual_climate_comparison_mu.png'

create_climate_comparison_plot(
    obs_station_means, 
    historical_spatial_means, 
    future_spatial_means,
    mask, 
    month_labels, 
    output_path
)

# Create counterfactual climatology maps
print("\nCreating counterfactual climate rainfall climatology maps...")
create_monthly_climatology_maps(
    future_interpolated_results,
    filename_suffix='counterfactual',
    vmin=0, vmax=400,
    colorbar_label='Rainfall Climatology [mm]'
)

print("\nAnalysis Summary:")
print("- Blue points: Historical climate (station vs spatial averages)")
print("- Orange points: Future climate with same rainfall means but changed correlations")
print("- Deviations from 1:1 line indicate station network adequacy changes under climate change")
print("- This isolates pure spatial correlation effects from mean rainfall changes")
print("- Climatology maps show spatial patterns under counterfactual future scenario")


Creating historical vs future climate comparison plot...
Saved comparison plot: figures/counterfactual_climate_comparison_mu.png

Creating counterfactual climate rainfall climatology maps...
Saved climatology map: figures/climatology_counterfactual.pdf

Analysis Summary:
- Blue points: Historical climate (station vs spatial averages)
- Orange points: Future climate with same rainfall means but changed correlations
- Deviations from 1:1 line indicate station network adequacy changes under climate change
- This isolates pure spatial correlation effects from mean rainfall changes
- Climatology maps show spatial patterns under counterfactual future scenario
